# ReAct (Reasoning + Acting)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/02-reasoning/15_react_reasoning_+_acting.ipynb)

**Category:** Reasoning & Logic  
**Technique #15**

---

## 📋 Description

ReAct (Reasoning + Acting) is a paradigm that interleaves reasoning traces with task-specific actions. The model alternates between thinking (reasoning about what to do) and acting (taking actions like searching, calculating, or using tools), creating a synergistic loop that improves decision-making.

**When to use:**
- Tasks requiring external information retrieval
- Multi-step problem solving with tool use
- Decision-making in dynamic environments
- Question answering that requires fact-checking
- Interactive task completion

## 🔧 How It Works

```
ReAct Loop:

┌─────────────────────────────────────────────────────┐
│ Question: "What is the elevation of Mount Everest  │
│           in meters, and who first reached it?"    │
└──────────────────┬──────────────────────────────────┘
                   │
    ┌──────────────┴──────────────┐
    ▼                             ▼
┌──────────┐              ┌──────────────┐
│ Thought  │              │    Action    │
│ "I need │─────────────▶│ Search[Mount │
│  to find│              │ Everest      │
│  info   │              │ elevation]   │
│  about  │              └──────┬───────┘
│ Everest │◀────────────────────┘
│ height" │              Observation:
└──────────┘              "8,849 meters"
    │                             │
    ▼                             ▼
┌──────────┐              ┌──────────────┐
│ Thought  │              │    Action    │
│ "Now I  │─────────────▶│ Search[first │
│ need to │              │ ascent Mount │
│ find who│              │ Everest]     │
│ first   │              └──────┬───────┘
│ climbed │◀────────────────────┘
│ it"     │              Observation:
└──────────┘              "Edmund Hillary,│
    │                      Tenzing Norgay"
    ▼                             │
┌─────────────────────────────────┐
│           Answer                │
│  Mount Everest is 8,849 meters  │
│  high. First ascent by Edmund   │
│  Hillary and Tenzing Norgay.    │
└─────────────────────────────────┘
```

**Pattern:** Thought → Action → Observation → (repeat) → Answer

In [ ]:
import osfrom getpass import getpass# Install required packages (uncomment if needed)# !pip install openai -q# Set up OpenAI API key securelyif "OPENAI_API_KEY" not in os.environ:    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")# Import OpenAIfrom openai import OpenAIclient = OpenAI()def get_completion(prompt, model="gpt-4", temperature=0.7):    """Helper function to get completions from OpenAI API"""    try:        response = client.chat.completions.create(            model=model,            messages=[                {"role": "system", "content": "You are a helpful assistant."},                {"role": "user", "content": prompt}            ],            temperature=temperature        )        return response.choices[0].message.content    except Exception as e:        return f"Error: {str(e)}"print("✅ Setup complete! Ready to experiment with prompts.")

## 💡 Basic Example

Implementing a simple ReAct agent for a multi-step question.

**Problem:** Find information about the Eiffel Tower's height and compare it to the Empire State Building.

In [ ]:
# ReAct (Reasoning + Acting) Implementationclass SimpleReActAgent:    def __init__(self):        self.memory = []        self.max_steps = 5    def search_knowledge(self, query):        """Simulated search function - in real use, this would call an API"""        knowledge_base = {            "eiffel tower height": "The Eiffel Tower is 330 meters (1,083 feet) tall.",            "empire state building height": "The Empire State Building is 443 meters (1,454 feet) tall including antenna.",            "eiffel tower built": "The Eiffel Tower was completed in 1889.",            "empire state building built": "The Empire State Building was completed in 1931."        }        for key, value in knowledge_base.items():            if query.lower() in key or key in query.lower():                return value        return f"[Simulated search result for: {query}]"    def calculate(self, expression):        """Simple calculator"""        try:            return str(eval(expression))        except:            return "Error in calculation"    def think_and_act(self, question):        """Main ReAct loop"""        print("="*60)        print("🤖 ReAct Agent")        print("="*60)        print(f"Question: {question}")        context = f"Question: {question}"        for step in range(self.max_steps):            print(f"--- Step {step + 1} ---")            # Generate Thought            thought_prompt = f"""{context}Based on the question and any previous observations, what should I think about next?Provide your thought in the format: Thought: [your reasoning]Thought:"""            thought_response = get_completion(thought_prompt, temperature=0.3)            thought = thought_response.replace("Thought:", "").strip()            print(f"🤔 Thought: {thought}")            # Generate Action            action_prompt = f"""{context}Thought: {thought}What action should I take? Choose from:- Search[query] - to search for information- Calculate[expression] - to perform math- Answer[answer] - when ready to provide final answerAction:"""            action_response = get_completion(action_prompt, temperature=0.3)            action = action_response.strip()            print(f"🔧 Action: {action}")            # Execute Action            if "Search[" in action:                query = action.split("[")[1].split("]")[0]                observation = self.search_knowledge(query)            elif "Calculate[" in action:                expr = action.split("[")[1].split("]")[0]                observation = self.calculate(expr)            elif "Answer[" in action or "answer:" in action.lower():                answer = action.split("[")[1].split("]")[0] if "[" in action else action                print(f"✅ Final Answer: {answer}")                return answer            else:                observation = "Invalid action format"            print(f"👁️ Observation: {observation}")            # Update context            context += f"Thought: {thought}Action: {action}Observation: {observation}"        print("⚠️ Reached max steps without answer")        return None# Test the agentagent = SimpleReActAgent()question = "How much taller is the Empire State Building than the Eiffel Tower?"agent.think_and_act(question)

## 🌍 Real-World Example

**Scenario:** Customer support agent that can look up information and provide solutions.

ReAct enables the agent to reason about customer issues and take appropriate actions.

In [ ]:
# Real-World: Customer Support ReAct Agentsupport_agent_prompt = """You are a customer support agent for an electronics company. Help the customer by thinking through their issue and taking appropriate actions.Available actions:- LookupOrder[order_id] - check order status- CheckWarranty[product] - check warranty information- SearchFAQ[query] - search knowledge base- Escalate[reason] - escalate to human agent- Solution[answer] - provide solution to customerCustomer: "I ordered a laptop 2 weeks ago (Order #12345) and it hasn't arrived. The tracking shows it's stuck in transit. What should I do?"Let's work through this step by step.Thought:"""print("="*60)print("🎧 Customer Support ReAct Agent")print("="*60)# Simulate the ReAct loopprint("📝 Simulated ReAct Trace:")react_trace = """Step 1:🤔 Thought: The customer has an order that hasn't arrived and is stuck in transit.    I should first look up their order to get more details.🔧 Action: LookupOrder[12345]👁️ Observation: Order #12345 - Laptop Pro X1, shipped 14 days ago,    last scan: 5 days ago at regional hub, status: delayedStep 2:🤔 Thought: The order is delayed at a regional hub for 5 days.    This is abnormal. I should check our FAQ for delayed shipment policies.🔧 Action: SearchFAQ[delayed shipment more than 5 days]👁️ Observation: For shipments delayed more than 5 days, customers are eligible    for a refund or replacement. Contact shipping partner for investigation.Step 3:🤔 Thought: The customer is eligible for a refund or replacement.    I should inform them of their options and offer to escalate for faster resolution.🔧 Action: Solution[I'm sorry for the delay. Your order has been stuck for 5 days,    which qualifies you for either a full refund or a replacement shipment.    Would you like me to: 1) Process a refund, 2) Send a replacement, or 3)    Escalate to our shipping team for urgent investigation?]"""print(react_trace)print("" + "="*60)print("🔄 ReAct Pattern Benefits:")print("="*60)print("✓ Transparent reasoning process")print("✓ Can use external tools and APIs")print("✓ Handles multi-step problems")print("✓ Can backtrack if observations contradict expectations")

## ⚠️ Failure Case

ReAct can fail when:
1. The action space is not well-defined
2. Observations are noisy or misleading
3. The model gets stuck in loops
4. Tool/API calls fail
5. The reasoning doesn't properly incorporate observations

In [ ]:
# Failure Case: ReAct limitationsprint("="*60)print("⚠️ ReAct Failure Cases")print("="*60)# Case 1: Loopingprint("❌ Case 1: Getting stuck in loops")print("-" * 40)loop_example = """Question: "What is the capital of France?"Bad ReAct trace:Step 1: Thought: I need to find the capital of France         Action: Search[capital of France]         Observation: Paris is the capital of FranceStep 2: Thought: Let me verify this information         Action: Search[Paris capital France]         Observation: Paris is the capital of FranceStep 3: Thought: Let me double-check         Action: Search[France capital city]         Observation: Paris is the capital of France... (continues looping)"""print(loop_example)print("💡 Solution: Add a counter and force answer after N steps")# Case 2: Wrong action selectionprint("❌ Case 2: Selecting inappropriate actions")print("-" * 40)wrong_action = """Question: "Calculate 15 + 27"Bad ReAct trace:Step 1: Thought: I need to calculate 15 + 27         Action: Search[15 + 27 calculation]         Observation: [Search results about addition]Step 2: Thought: Let me search for a calculator         Action: Search[online calculator]         Observation: [List of calculator websites]💡 Problem: Should have used Calculate[15 + 27] directly!"""print(wrong_action)print("" + "="*60)print("💡 Best Practices to Avoid Failures:")print("="*60)print("1. Define clear action space")print("2. Set maximum step limits")print("3. Include examples in prompt")print("4. Validate observations make sense")print("5. Use structured output formats")

## 📊 Benchmark

| Task | Standard Prompting | CoT | ReAct |
|------|-------------------|-----|-------|
| HotPotQA (QA) | 28.3% | 35.1% | 42.1% |
| FEVER (Fact Check) | 71.8% | 75.2% | 82.4% |
| WebShop (E-commerce) | 28.7% | 31.5% | 44.8% |
| ALFWorld (Decision Making) | 45.0% | 48.2% | 71.6% |

**Key Findings:**
- ReAct significantly outperforms on interactive tasks
- Synergy between reasoning and acting is crucial
- Most effective when tools/actions are well-defined
- Can be combined with other techniques (CoT, self-consistency)

## 🎮 Interactive Playground

Experiment with your own prompts below!

In [ ]:
# 🎮 Interactive Playground# Modify the prompt below and run to see resultsyour_prompt = """# Your prompt here"""# Get responseresponse = get_completion(your_prompt)print("="*50)print("📝 Response:")print("="*50)print(response)

## 💡 Tips & Tricks

**Implementation Tips:**
- ✓ Define a clear, limited action space
- ✓ Provide few-shot examples of the ReAct pattern
- ✓ Set maximum iteration limits
- ✓ Make observations informative and structured
- ✓ Include error handling for failed actions
- ✗ Don't allow unlimited steps
- ✗ Don't make action space too large
- ✗ Don't skip the reasoning (Thought) step

**Common Action Types:**
- Search[query] - Information retrieval
- Calculate[expr] - Mathematical computation
- Lookup[key] - Database queries
- API[endpoint] - External service calls
- Answer[response] - Final answer

## 📚 References

1. **ReAct: Synergizing Reasoning and Acting in Language Models** (Yao et al., 2022)
   - [Paper](https://arxiv.org/abs/2210.03629)

2. **WebGPT: Browser-assisted question-answering** (Nakano et al., 2021)
   - [Paper](https://arxiv.org/abs/2112.09332)

3. **LangChain Documentation: ReAct**
   - [Docs](https://python.langchain.com/docs/modules/agents/agent_types/react)